In [ ]:
# NOTEBOOK NAME
# SimpleCFADs.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

from matplotlib.colors import Normalize

In [ ]:
# CFAD LOOP
# ADDED 2026-06-10T18:56UTC+10:00

# what does the radar data use for the "fill value"?
FillValue = -32

# CHOOSE YOUR CFAD TYPE ('Absolute' or 'Relative')
Version = 'Absolute'

# CHOOSE YOUR RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# name the radar site
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# CHOOSE YOUR DAY
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 25
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# CHOOSE YOUR QUALITY CONTROL SETTINGS
# # taken from Aragon et al. 2024
# MinValidZDR = -4 # ZDR DOESNT WORK FOR MACKAY EARLY 2024 BECAUSE OF THE SOURCE RADAR DATA IN STORAGE
# MaxValidZDR =  4
MinValidRhoHV = 0.85

VarName     = 'Reflectivity'
VarNameLong = 'corrected_reflectivity'

# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)

# write out the data in one string
RadarFileDate  = YYYY + MM + DD 

# CHOOSE YOUR SOUNDING DATA
Ztime = '00'
UpperAirSiteID = '95282'
# 95282 for Townsville, 
# 94299 for Willis Island

# CHOOSE YOUR ANNULUS OF CONSIDERATION
InnerRadius =  3  # [km] (inner radius of the annulus of consideration)
OuterRadius = 50  # [km] (outer radius of the annulus of consideration)

# CHOOSE YOUR HEIGHTS OF CONSIDERATION
MinHeight = 0.5 # [km] minimum height in the CFAD
MaxHeight = 15 # [km] maximum height in the CFAD

# CHOOSE YOUR HOUR and MINUTE
# houri = 00
# mini = 00

# sounding data follow-on and loading
SoundingFolder = '/home/563/sg3241/TownsvilleSoundings/' 
SoundingFileName = RadarFileDate + Ztime + '-' + UpperAirSiteID + '.csv'
SoundingPath = SoundingFolder + SoundingFileName

SoundingData = pd.read_csv(SoundingPath)

# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60

    # add leading zeros for strings
    YYYY = str(RadarYear).zfill(4)
    MM = str(RadarMonth).zfill(2)
    DD = str(RadarDay).zfill(2)
    
    # write out the data in one string
    RadarFileDate  = YYYY + MM + DD 
    
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
    NetCDFstorageFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile 
                            
    try:
        xgrid = xr.open_dataset(NetCDFstoragePath)
        print('working on reading the file for ' + RadarFileTimePrint)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # calculate the distance from the radar and add it to the data frame as a new variable for data selection purposes
    xgrid['distance'] = distance = np.sqrt(xgrid['x']**2 + xgrid['y']**2)
    xgrid['distance'].attrs = {'long_name': 'Horizontal distance from radar', 'units': 'm'}
    
    ConsideredHeights = xgrid.z[ np.where( (xgrid.z>=MinHeight*1000) & (xgrid.z<=MaxHeight*1000) ) ]
    
    MinDBZ = -10 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
    MaxDBZ = 60 # [dBZ] minimum value of reflectivity you want to consider in the CFADs
    
    NumHeights = np.size(ConsideredHeights) # the total number of altitudes in consideration
    
    # find a floor and ceiling to the values of DBZ
    LoEndDBZ = MinDBZ #int(np.floor(np.nanmin(flatZs)))
    HiEndDBZ = MaxDBZ #int(np.ceil(np.nanmax(flatZs)))
    
    NumDBZs = HiEndDBZ - LoEndDBZ # the total number of reflectivity [bins] in consideration

    # ROUGH QUALITY CONTROL SECTION
    # create a mask only where these quality control condtions are met
    ConditionGridQCA = xgrid['corrected_cross_correlation_ratio'] > MinValidRhoHV   # (y, x) boolean masks
    
    # I WOULD LIKE TO ADD CONDITIONS WITH DIFFERENTIAL REFLECTIVITY, BUT THIS RADAR DOES NOT HAVE VALID DATA YET
    # ConditionGridB = xgrid['corrected_differential_reflectivity'] > MinValidZDR
    # ConditionGridC = xgrid['corrected_differential_reflectivity'] < MaxValidZDR  
    
    ConditionGridQC = ConditionGridQCA #* ConditionGridB * ConditionGridC   # combined boolean mask

    # mask the reflectivities over only the specified annulus
    ConditionGridA = xgrid['distance'] > InnerRadius * 1000   # (y, x) boolean mask (converting from km to m)
    ConditionGridB = xgrid['distance'] < OuterRadius * 1000   # (y, x) boolean mask (converting from km to m)
    ConditionGridC = xgrid['z'] >= MinHeight * 1000   # (y, x) boolean mask (converting from km to m)
    ConditionGridD = xgrid['z'] <= MaxHeight * 1000   # (y, x) boolean mask (converting from km to m)

    #  # apply the condtional mask to the variable array before plotting
    # ValidVariableArray = xgrid[VarNameLong].where(ConditionGrid)
    # PlottingArray = ValidVariableArray[0,:,EWslicei,:]

    # # REAL DATA PLOTTING
    # # REAL DATA PLOTTING
    # # REAL DATA PLOTTING
    # fig, ax = plt.subplots(figsize=(8,6))
    # GridViewer = pcolormeshC(lon_slice, xgrid.z*0.001, PlottingArray, ax=ax, cmap=VarColourBar, norm=VarColourBar_norm)
    #                                          # mutiply by 0.001 to get distances in km
    
    ConditionGrid = ConditionGridA * ConditionGridB * ConditionGridC * ConditionGridD * ConditionGridQC # combined boolean mask
    masked_reflectivity = xgrid['corrected_reflectivity'].where(ConditionGrid)  # MASK THE Z VALUES
    
    # create an empty array for storing frequencies and relative frequencies of reflectivity values
    AbsCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
    NormCountsGrid = np.full([NumDBZs,NumHeights], np.nan)
    
    # how many total grid cells in the radar data
    # TotalCells = np.size(xgrid.corrected_reflectivity)
    TotalCells = float(np.sum(ConditionGrid)) # the total number of cells in consideration
    
    # loop through each height and collect the relative frequencies
    for zi in range(0, NumHeights):
    
        # store the flattened array of DBZ values
        flatZs = np.ndarray.flatten(np.array(masked_reflectivity[0,zi,:,:]))
        
        # replace the fill value with nans
        flatZs = np.where(flatZs == FillValue, np.nan, flatZs)
        
        # replace the low value with nans
        flatZs = np.where(flatZs < MinDBZ, np.nan, flatZs)

        
        # retrieve the data for a histogram
        counts, bins = np.histogram(flatZs[~np.isnan(flatZs)], bins=HiEndDBZ-LoEndDBZ, range=[LoEndDBZ, HiEndDBZ])
    
        # calculate the total cell frequencies and frequencies for that height
        AbsCounts  = counts * (1/TotalCells)
        NormCounts = counts * (1/np.sum(counts))
        
        # store away those counts and normalised counts in the grand CFAD data
        AbsCountsGrid[:,zi]  = AbsCounts
        NormCountsGrid[:,zi] = NormCounts
    
    
    # start plotting!
    
    # based on the user's choice of version, pick the variable to plot, name the units its in, and set the colourbar limits for that variable
    if (Version == 'Absolute'):
        PlotCounts = AbsCountsGrid * 100 # multiply by 100 to make it a percentage
        PlotUnit = '% of considered grid cells'
        MinVarVal = 0.00
        MaxVarVal = 1.50
    elif (Version == 'Relative'):
        PlotCounts = NormCountsGrid  
        PlotUnit = 'per dBZ per km'
        MinVarVal = 0.00
        MaxVarVal = 0.40
    else:
        sys.exit("please enter 'Absolute' or 'Relative' for Version")
    
    
    # intervals of DBZ bins on the plot
    DBZspacing = 1 # [dBZ]
    
    DBZs = np.arange(LoEndDBZ,HiEndDBZ,DBZspacing) # (X COORDINATE ON THE PLOT) create a list of all possible DBZ values from -39 to 100 in steps of 1 
    Heights = np.array(ConsideredHeights) * 0.001  # (Y COORDINATE ON THE PLOT) [converted to km] create a list of all of the altitudes where data are stored
    
    # intervals of altitudes on the plot
    HeightSpacing = 0.5 # [km] 
    
    UprightFrequencies = np.transpose(PlotCounts) # (VALUES ON THE PLOT) transpose the stored relative frequencies of the reflectivities
    
    # normalise frequencies to per unit DBZ per km of height
    NormUprightFrequencies = UprightFrequencies * (1/DBZspacing) * (1/HeightSpacing)  
    
    # PLOT A CFAD! (sort of)
    fig, ax = plt.subplots(figsize=(8,6))

    CFAD1 = pcolormeshC(DBZs , Heights, NormUprightFrequencies, ax=ax, cmap='nipy_spectral', vmin=MinVarVal, vmax=MaxVarVal)
    plt.colorbar(CFAD1, ax=ax, label = Version + ' Frequency [' + PlotUnit + ']')


    # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
    # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
    # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
    TropopauseAlt = 16.8
    # plot the tropopause
    ax.axhline(y=TropopauseAlt, color=[0.6,0.0,0.0], linewidth=1, label='Tropopause')

    # plot the bottom and top of the Dentritic Growth Zone Temperature Range
    ax.axhline(y=temp_crossing_altitude(SoundingData, -10), color=[0.0,0.8,0.8], linewidth=1, label='Dentritic Growth Zone Temperature Range (-10 to -20°C)')
    ax.axhline(y=temp_crossing_altitude(SoundingData, -20), color=[0.0,0.8,0.8], linewidth=1)

    # plot the bottom and top of the Hallett-Mossop Temperature Range
    ax.axhline(y=temp_crossing_altitude(SoundingData, -3), color=[0.8,0.4,0.0], linewidth=1, label='Hallett-Mossop Temperature Range (-8 to -3°C)')
    ax.axhline(y=temp_crossing_altitude(SoundingData, -8), color=[0.8,0.4,0.0], linewidth=1)

    # plot the freezing level
    ax.axhline(y=temp_crossing_altitude(SoundingData, 0), color='blue', linewidth=1, label='Freezing Level (0°C)')

    ax.legend(loc='upper right')
    
    plt.xlim([MinDBZ, MaxDBZ])
    plt.ylim([0, MaxHeight+3])
    
    plt.title(Version + ' Frequencies of Reflectivity Values (ρHV > ' + str(MinValidRhoHV) + ') by Altitude\n for ' + RadarSiteName + ' Radar on ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC\n' + \
             'For the Annulus between ' + str(InnerRadius) + ' and ' + str(OuterRadius) + ' km from the Radar') 
    
    ax.set_xlabel('Corrected Reflectivity [dBZ]')
    ax.set_ylabel('Altitude [km]')

    plt.grid(linewidth=0.5)
    
    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/' + VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '/'
    SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarName + '_' + Version + 'CFAD.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('Creating Folder: ' + SaveFolder)
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    plt.close()

In [ ]:
# GIF MAKER
# FOR CFADS

SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/' + VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '/'
                
# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named

images = [Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(VarName + '_' + Version + 'CFAD.png')]

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/CFAD/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + VarName + '_' + Version + 'CFAD' + '_RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(GIFsavePath, save_all=True, append_images=images[1:], duration=200, loop=0)
                                                                    # ms per frame    # 0 = loop forever

print('Saved GIF for ' + RadarFileDate)